# Masked Residual Restoration

This notebook trains highlight removal models without hard-hole inpainting.

- U-Net variants see the original RGB image, and optionally `1 - soft_mask` as an extra input channel.
- Partial-conv models receive the original RGB image plus a binary validity mask, so masking happens inside the partial-conv layers.
- All models predict an RGB residual.
- The final correction is applied only inside the binary hole: `y_pred = x + hole_mask * residual`.
- Optimization is measured on the masked region by default.

Note: the partial-conv models still use standard partial-convolution semantics. They do not rewrite kernels to sparse hole-only compute.


In [ ]:
import sys
from pathlib import Path
import importlib
import time
from collections import defaultdict

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import clear_output
from torch.utils.data import DataLoader, Subset

sys.path.append("/Users/27171653/Desktop/PhD/Specular-Highlights/sh_models")

from pipeline import PSDloader as psd_loader
from sh_models import pconvAE as pc_hr

three_channel_unet = importlib.import_module("3chanelunet")
pconv_unet = importlib.import_module("pconvUnet")
prepo_pd_resnet = importlib.reload(importlib.import_module("prepo.models.pd_resnet"))

diffuse_images = "/Users/27171653/Desktop/PhD/Specular-Highlights/data/train/can/diffuse"
specular_images = "/Users/27171653/Desktop/PhD/Specular-Highlights/data/train/can/glossy"

dataset, loader = psd_loader.make_psd_dataloader(
    diffuse_dir=diffuse_images,
    specular_dir=specular_images,
    batch_size=4,
    soft_gamma=1.0,
    threshold_method="quantile",
    threshold=0.7,
)

small_dataset = Subset(dataset, [6])
small_loader = DataLoader(
    small_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
weights_dir = Path("weights")
weights_dir.mkdir(parents=True, exist_ok=True)

TEST_EPOCHS = 100


In [ ]:
def _move_batch_to_device(batch, device):
    out = {}
    for key, value in batch.items():
        out[key] = value.to(device, non_blocking=True) if torch.is_tensor(value) else value
    return out


def _get_loss_fn(loss_name="l1", reduction="mean"):
    loss_name = loss_name.lower()
    if loss_name == "l1":
        return nn.L1Loss(reduction=reduction)
    if loss_name == "mse":
        return nn.MSELoss(reduction=reduction)
    if loss_name == "smoothl1":
        return nn.SmoothL1Loss(reduction=reduction)
    raise ValueError("loss_name must be 'l1', 'mse', or 'smoothl1'")


def _pick_mask_tensor(batch, mask_source="soft_mask"):
    if mask_source == "mask" and "mask" in batch:
        return batch["mask"]
    if mask_source == "soft_mask" and "soft_mask" in batch:
        return batch["soft_mask"]
    if "soft_mask" in batch:
        return batch["soft_mask"]
    if "mask" in batch:
        return batch["mask"]
    raise KeyError("Batch must contain either 'mask' or 'soft_mask'.")


def _prepare_restoration_masks(batch, *, mask_source="soft_mask", valid_threshold=0.95):
    raw_valid_mask = _pick_mask_tensor(batch, mask_source).float().clamp(0.0, 1.0)
    valid_mask = (raw_valid_mask >= valid_threshold).float()
    hole_mask = 1.0 - valid_mask
    corruption_map = 1.0 - raw_valid_mask
    return raw_valid_mask, valid_mask, hole_mask, corruption_map


def _masked_mean(pixel_values, mask, eps=1e-8):
    if pixel_values.ndim != 4:
        raise ValueError(f"Expected pixel_values [B,C,H,W], got {tuple(pixel_values.shape)}")
    if mask.ndim != 4:
        raise ValueError(f"Expected mask [B,1,H,W], got {tuple(mask.shape)}")
    if mask.shape[1] != 1:
        mask = mask.mean(dim=1, keepdim=True)
    denom = mask.sum() * pixel_values.shape[1]
    return (pixel_values * mask).sum() / (denom + eps)


def _forward_masked_residual_model(model, x, valid_mask, hole_mask, soft_mask=None):
    try:
        residual = model(
            x,
            valid_mask=valid_mask,
            hole_mask=hole_mask,
            soft_mask=soft_mask,
        )
    except TypeError:
        try:
            residual = model(x, valid_mask=valid_mask, hole_mask=hole_mask)
        except TypeError:
            residual = model(x, valid_mask)
    if residual.shape != x.shape:
        raise ValueError(
            f"Residual and input shapes must match, got {tuple(residual.shape)} and {tuple(x.shape)}"
        )
    return residual


def _select_correction_mask(hole_mask, corruption_map, correction_source="hole"):
    if correction_source == "hole":
        return hole_mask
    if correction_source == "soft":
        return corruption_map
    raise ValueError("correction_source must be 'hole' or 'soft'")


def _run_masked_residual_batch(
    model,
    batch,
    pixel_loss_fn,
    *,
    mask_source="soft_mask",
    valid_threshold=0.95,
    correction_source="hole",
    optimize_source="hole",
):
    x = batch["input"].float()
    y_true = batch["target"].float()

    if x.ndim != 4 or x.shape[1] != 3:
        raise ValueError(f"Expected input shape [B,3,H,W], got {tuple(x.shape)}")
    if y_true.ndim != 4 or y_true.shape[1] != 3:
        raise ValueError(f"Expected target shape [B,3,H,W], got {tuple(y_true.shape)}")

    raw_valid_mask, valid_mask, hole_mask, corruption_map = _prepare_restoration_masks(
        batch,
        mask_source=mask_source,
        valid_threshold=valid_threshold,
    )

    residual = _forward_masked_residual_model(
        model,
        x,
        valid_mask,
        hole_mask,
        soft_mask=raw_valid_mask,
    )

    correction_mask = _select_correction_mask(hole_mask, corruption_map, correction_source=correction_source)
    y_pred = x + correction_mask * residual

    pixel_loss = pixel_loss_fn(y_pred, y_true)
    if optimize_source == "hole":
        loss = _masked_mean(pixel_loss, hole_mask)
    elif optimize_source == "soft":
        loss = _masked_mean(pixel_loss, corruption_map)
    else:
        raise ValueError("optimize_source must be 'hole' or 'soft'")

    abs_err = (y_pred - y_true).abs()

    return {
        "loss": loss,
        "x": x,
        "y_true": y_true,
        "y_pred": y_pred,
        "residual": residual,
        "raw_valid_mask": raw_valid_mask,
        "valid_mask": valid_mask,
        "hole_mask": hole_mask,
        "corruption_map": corruption_map,
        "correction_mask": correction_mask,
        "hole_l1": _masked_mean(abs_err, hole_mask),
        "soft_region_l1": _masked_mean(abs_err, corruption_map),
        "context_l1": _masked_mean(abs_err, valid_mask),
        "full_l1": abs_err.mean(),
        "identity_hole_l1": _masked_mean((x - y_true).abs(), hole_mask),
    }


@torch.no_grad()
def evaluate_masked_residual_model(
    model,
    loader,
    *,
    loss_name="l1",
    device=None,
    mask_source="soft_mask",
    valid_threshold=0.95,
    correction_source="hole",
    optimize_source="hole",
):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()
    pixel_loss_fn = _get_loss_fn(loss_name, reduction="none")
    running = defaultdict(float)
    n_samples = 0

    for batch in loader:
        batch = _move_batch_to_device(batch, device)
        outputs = _run_masked_residual_batch(
            model,
            batch,
            pixel_loss_fn,
            mask_source=mask_source,
            valid_threshold=valid_threshold,
            correction_source=correction_source,
            optimize_source=optimize_source,
        )

        bs = outputs["x"].shape[0]
        n_samples += bs
        for key in ("loss", "hole_l1", "soft_region_l1", "context_l1", "full_l1", "identity_hole_l1"):
            running[key] += outputs[key].item() * bs

    return {key: value / max(n_samples, 1) for key, value in running.items()}


@torch.no_grad()
def preview_masked_residual_samples(
    model,
    preview_dataset,
    *,
    device=None,
    max_items=3,
    loss_name="l1",
    mask_source="soft_mask",
    valid_threshold=0.95,
    correction_source="hole",
    optimize_source="hole",
):
    if preview_dataset is None or len(preview_dataset) == 0:
        return

    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    pixel_loss_fn = _get_loss_fn(loss_name, reduction="none")
    rows = min(max_items, len(preview_dataset))
    fig, axes = plt.subplots(rows, 6, figsize=(24, 4 * rows))
    if rows == 1:
        axes = axes[None, :]

    model.eval()
    for row in range(rows):
        sample = preview_dataset[row]
        batch = {
            key: value.unsqueeze(0).to(device) if torch.is_tensor(value) else value
            for key, value in sample.items()
        }
        outputs = _run_masked_residual_batch(
            model,
            batch,
            pixel_loss_fn,
            mask_source=mask_source,
            valid_threshold=valid_threshold,
            correction_source=correction_source,
            optimize_source=optimize_source,
        )

        inp = outputs["x"][0].detach().cpu().permute(1, 2, 0).clamp(0, 1)
        tgt = outputs["y_true"][0].detach().cpu().permute(1, 2, 0).clamp(0, 1)
        pred = outputs["y_pred"][0].detach().cpu().permute(1, 2, 0).clamp(0, 1)
        residual = outputs["residual"][0].detach().cpu().permute(1, 2, 0)
        hole_mask = outputs["hole_mask"][0, 0].detach().cpu()
        correction = outputs["correction_mask"][0, 0].detach().cpu()
        abs_err = (outputs["y_pred"][0].detach().cpu() - outputs["y_true"][0].detach().cpu()).abs().mean(dim=0)

        axes[row, 0].imshow(inp)
        axes[row, 0].set_title(f"Input\n{sample.get('name', f'sample_{row}')}")
        axes[row, 1].imshow(tgt)
        axes[row, 1].set_title("Target")
        axes[row, 2].imshow(pred)
        axes[row, 2].set_title(f"Prediction\nHole MAE {outputs['hole_l1'].item():.4f}")
        axes[row, 3].imshow(hole_mask, cmap="gray", vmin=0, vmax=1)
        axes[row, 3].set_title("Binary hole")
        axes[row, 4].imshow(correction, cmap="gray", vmin=0, vmax=1)
        axes[row, 4].set_title("Applied correction mask")
        axes[row, 5].imshow(abs_err, cmap="gray", vmin=0, vmax=1)
        axes[row, 5].set_title("Mean abs error")

        for col in range(6):
            axes[row, col].axis("off")

    plt.tight_layout()
    plt.show()


def plot_masked_residual_history(history, model_label="model"):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs, history["train_loss"], label="train")
    if history.get("val_loss"):
        axes[0].plot(epochs, history["val_loss"], label="val")
    axes[0].set_title(f"{model_label} loss")
    axes[0].set_xlabel("epoch")
    axes[0].set_ylabel("loss")
    axes[0].legend()

    axes[1].plot(epochs, history["train_hole_l1"], label="train hole_l1")
    axes[1].plot(epochs, history["train_identity_hole_l1"], label="identity hole_l1")
    if history.get("val_hole_l1"):
        axes[1].plot(epochs, history["val_hole_l1"], label="val hole_l1")
        if history.get("val_identity_hole_l1"):
            axes[1].plot(epochs, history["val_identity_hole_l1"], label="val identity")
    axes[1].set_title(f"{model_label} masked MAE")
    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("L1")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


def train_masked_residual_model(
    model,
    train_loader,
    val_loader=None,
    *,
    lr=1e-3,
    weight_decay=0.0,
    epochs=TEST_EPOCHS,
    device=None,
    loss_name="l1",
    grad_clip=None,
    log_every=10,
    plot_every=None,
    preview_dataset=None,
    preview_every=0,
    save_best_path=None,
    model_label="Masked residual model",
    mask_source="soft_mask",
    valid_threshold=0.95,
    correction_source="hole",
    optimize_source="hole",
):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    plot_every = plot_every or epochs
    model = model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    pixel_loss_fn = _get_loss_fn(loss_name, reduction="none")
    history = defaultdict(list)
    best_loss = float("inf")
    start_time = time.time()

    def _save_checkpoint(path, epoch, train_metrics, val_metrics=None):
        ckpt = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_metrics": train_metrics,
            "config": {
                "lr": lr,
                "weight_decay": weight_decay,
                "loss_name": loss_name,
                "mask_source": mask_source,
                "valid_threshold": valid_threshold,
                "correction_source": correction_source,
                "optimize_source": optimize_source,
                "model_label": model_label,
            },
        }
        if val_metrics is not None:
            ckpt["val_metrics"] = val_metrics
        torch.save(ckpt, path)

    for epoch in range(1, epochs + 1):
        model.train()
        running = defaultdict(float)
        n_samples = 0

        for batch in train_loader:
            batch = _move_batch_to_device(batch, device)
            optimizer.zero_grad(set_to_none=True)

            outputs = _run_masked_residual_batch(
                model,
                batch,
                pixel_loss_fn,
                mask_source=mask_source,
                valid_threshold=valid_threshold,
                correction_source=correction_source,
                optimize_source=optimize_source,
            )
            loss = outputs["loss"]
            loss.backward()

            if grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

            optimizer.step()

            bs = outputs["x"].shape[0]
            n_samples += bs
            for key in ("loss", "hole_l1", "soft_region_l1", "context_l1", "full_l1", "identity_hole_l1"):
                running[key] += outputs[key].item() * bs

        train_metrics = {key: value / max(n_samples, 1) for key, value in running.items()}
        history["train_loss"].append(train_metrics["loss"])
        history["train_hole_l1"].append(train_metrics["hole_l1"])
        history["train_soft_region_l1"].append(train_metrics["soft_region_l1"])
        history["train_context_l1"].append(train_metrics["context_l1"])
        history["train_full_l1"].append(train_metrics["full_l1"])
        history["train_identity_hole_l1"].append(train_metrics["identity_hole_l1"])

        val_metrics = None
        monitor_loss = train_metrics["loss"]
        if val_loader is not None:
            val_metrics = evaluate_masked_residual_model(
                model,
                val_loader,
                loss_name=loss_name,
                device=device,
                mask_source=mask_source,
                valid_threshold=valid_threshold,
                correction_source=correction_source,
                optimize_source=optimize_source,
            )
            history["val_loss"].append(val_metrics["loss"])
            history["val_hole_l1"].append(val_metrics["hole_l1"])
            history["val_soft_region_l1"].append(val_metrics["soft_region_l1"])
            history["val_context_l1"].append(val_metrics["context_l1"])
            history["val_full_l1"].append(val_metrics["full_l1"])
            history["val_identity_hole_l1"].append(val_metrics["identity_hole_l1"])
            monitor_loss = val_metrics["loss"]

        if save_best_path and monitor_loss < best_loss:
            best_loss = monitor_loss
            _save_checkpoint(save_best_path, epoch, train_metrics, val_metrics)

        if epoch == 1 or epoch % log_every == 0 or epoch == epochs:
            elapsed = time.time() - start_time
            message = (
                f"[{model_label}] epoch {epoch:03d}/{epochs:03d} | "
                f"loss {train_metrics['loss']:.6f} | "
                f"hole_l1 {train_metrics['hole_l1']:.6f} | "
                f"identity_hole_l1 {train_metrics['identity_hole_l1']:.6f} | "
                f"elapsed {elapsed/60.0:.1f} min"
            )
            if val_metrics is not None:
                message += f" | val_hole_l1 {val_metrics['hole_l1']:.6f}"
            print(message)

        if preview_every and preview_dataset is not None and (epoch % preview_every == 0 or epoch == epochs):
            preview_masked_residual_samples(
                model,
                preview_dataset,
                device=device,
                loss_name=loss_name,
                mask_source=mask_source,
                valid_threshold=valid_threshold,
                correction_source=correction_source,
                optimize_source=optimize_source,
            )

        if plot_every and (epoch % plot_every == 0 or epoch == epochs):
            clear_output(wait=True)
            plot_masked_residual_history(history, model_label=model_label)

    return model, history


In [ ]:
class _PDResNetDecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.up = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
        fused_channels = out_channels + skip_channels
        self.fuse = nn.Sequential(
            nn.Conv2d(fused_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x, skip=None):
        x = self.up(x)
        if skip is not None:
            if x.shape[-2:] != skip.shape[-2:]:
                x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            x = torch.cat([x, skip], dim=1)
        return self.fuse(x)


class PrepoPDResNetResidualAdapter(nn.Module):
    def __init__(self, out_channels=3):
        super().__init__()
        self.backbone = prepo_pd_resnet.pdresnet18(pretrained=False, num_classes=out_channels)
        self.dec4 = _PDResNetDecoderBlock(512, 256, 256)
        self.dec3 = _PDResNetDecoderBlock(256, 128, 128)
        self.dec2 = _PDResNetDecoderBlock(128, 64, 64)
        self.dec1 = _PDResNetDecoderBlock(64, 64, 64)
        self.dec0 = _PDResNetDecoderBlock(64, 0, 32)
        self.out = nn.Conv2d(32, out_channels, kernel_size=1)

    def forward(self, x, valid_mask=None, hole_mask=None, soft_mask=None):
        if valid_mask is None:
            raise ValueError("PrepoPDResNetResidualAdapter requires valid_mask.")
        (stem, enc1, enc2, enc3, enc4), _ = self.backbone.forward_encoder(x, valid_mask)
        y = self.dec4(enc4, enc3)
        y = self.dec3(y, enc2)
        y = self.dec2(y, enc1)
        y = self.dec1(y, stem)
        y = self.dec0(y)
        if y.shape[-2:] != x.shape[-2:]:
            y = F.interpolate(y, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return self.out(y)


class PConvUNetResidualAdapter(nn.Module):
    def __init__(self, in_channels=3, base_channels=16, depth=3, out_channels=3):
        super().__init__()
        self.model = pconv_unet.UNet(
            in_channels=in_channels,
            base_channels=base_channels,
            depth=depth,
            out_channels=out_channels,
            Attention=None,
        )

    def forward(self, x, valid_mask=None, hole_mask=None, soft_mask=None):
        if valid_mask is None:
            raise ValueError("PConvUNetResidualAdapter requires valid_mask.")
        return self.model(x, valid_mask)


class ThreeChannelUNetResidualAdapter(nn.Module):
    def __init__(
        self,
        *,
        in_channels=3,
        base_channels=16,
        depth=3,
        out_channels=3,
        bottleneck_length=1,
        num_heads=4,
        use_bottleneck_self_attention=True,
        use_soft_mask_input=False,
    ):
        super().__init__()
        self.use_soft_mask_input = use_soft_mask_input
        self.model = three_channel_unet.UNet(
            in_channels=in_channels + (1 if use_soft_mask_input else 0),
            base_channels=base_channels,
            depth=depth,
            out_channels=out_channels,
            Attention=(three_channel_unet.TransformerEncoderSA if use_bottleneck_self_attention else None),
            OnlyonBottleneck=use_bottleneck_self_attention,
            num_heads=num_heads,
            bottleneck_length=bottleneck_length,
        )

    def forward(self, x, valid_mask=None, hole_mask=None, soft_mask=None):
        if self.use_soft_mask_input:
            if soft_mask is None:
                raise ValueError("ThreeChannelUNetResidualAdapter with use_soft_mask_input=True requires soft_mask.")
            corruption_map = 1.0 - soft_mask
            x = torch.cat([x, corruption_map], dim=1)
        return self.model(x)


MODEL_BUILDERS = {
    "unet3_residual": lambda: ThreeChannelUNetResidualAdapter(use_soft_mask_input=False),
    "unet4_residual_softmask": lambda: ThreeChannelUNetResidualAdapter(use_soft_mask_input=True),
    "pconv_autoencoder_residual": lambda: pc_hr.HighlightRemovalAutoencoder(
        in_channels=3,
        out_channels=3,
        base_features=64,
        depth=4,
        max_features=512,
        bottleneck_blocks=2,
        use_conv_shift=False,
    ),
    "pconv_unet_residual": lambda: PConvUNetResidualAdapter(),
    "pdresnet_residual": lambda: PrepoPDResNetResidualAdapter(out_channels=3),
}


MODEL_SAVE_NAMES = {
    "unet3_residual": "3channel_unet_bottleneck_self_attention_masked_residual_rgb.pth",
    "unet4_residual_softmask": "4channel_unet_bottleneck_self_attention_masked_residual_rgb.pth",
    "pconv_autoencoder_residual": "pconv_autoencoder_masked_residual_rgb.pth",
    "pconv_unet_residual": "pconv_unet_masked_residual_rgb.pth",
    "pdresnet_residual": "pdresnet_masked_residual_rgb.pth",
}


In [ ]:
# Choose one model and train it with masked residual correction.
# Available model_name values:
# - unet3_residual
# - unet4_residual_softmask
# - pconv_autoencoder_residual
# - pconv_unet_residual
# - pdresnet_residual

model_name = "unet4_residual_softmask"

model = MODEL_BUILDERS[model_name]()

trained_model, history = train_masked_residual_model(
    model=model,
    train_loader=loader,
    lr=1e-3,
    epochs=TEST_EPOCHS,
    device=device,
    loss_name="l1",
    log_every=10,
    plot_every=TEST_EPOCHS,
    preview_dataset=small_dataset,
    preview_every=0,
    save_best_path=str(weights_dir / MODEL_SAVE_NAMES[model_name]),
    model_label=model_name,
    mask_source="soft_mask",
    valid_threshold=0.95,
    correction_source="hole",
    optimize_source="hole",
)

print(f"Final {model_name} train loss: {history['train_loss'][-1]:.6f}")
print(f"Final {model_name} train hole_l1: {history['train_hole_l1'][-1]:.6f}")
print(f"Final {model_name} train identity_hole_l1: {history['train_identity_hole_l1'][-1]:.6f}")


In [ ]:
# Optional: preview the trained model on the quick sample set after training.

preview_masked_residual_samples(
    trained_model,
    preview_dataset=small_dataset,
    device=device,
    max_items=1,
    loss_name="l1",
    mask_source="soft_mask",
    valid_threshold=0.95,
    correction_source="hole",
    optimize_source="hole",
)
